In [1]:
# In the previous lesson we built a RAG pipeline with RAGBase.rag() and saw it fail on the "Olama" typo. The search returned nothing useful, and the LLM had no way to recover.

# The flow that broke:

In [2]:
# The pipeline is fixed: search, build prompt, LLM.

# def rag(self, query):
#     search_results = self.search(query)
#     prompt = self.build_prompt(query, search_results)
#     answer = self.llm(prompt)
#     return answer

# The LLM is a passenger here, not a driver. It never sees the bad search results, so it can't try again with a corrected query.

In [11]:
# The difference is about who makes the decisions:

#     With RAG, the developer decides. We fix the steps up front, so search always runs once with the exact user query.
#     With an agent, the LLM decides. It chooses which actions to take and when to stop.

# The mechanism that makes this possible is function calling, and that's what the rest of this lesson is about.

In [1]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [2]:
MODEL = "openai/gpt-oss-20b"


In [3]:
# ASKING WITHOUT TOOLS,
message1 = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=message1,
)

response.choices[0].message.content

'Sure thing! I can help you get started. To see if you can enroll right now, I’ll need a few details:\n\n1. **Full name**  \n2. **Student or employee ID** (if applicable)  \n3. **Email address**  \n4. **Preferred start date** (if you have one in mind)\n\nOnce I have those, I’ll check the course schedule and let you know if there’s a spot available or if you’ll need to wait for the next cohort. Feel free to send me the info whenever you’re ready!'

In [6]:
from ingest import load_faq_data, build_index

# Load the documents and build the index
docs = load_faq_data()
index = build_index(docs)


SSLError: HTTPSConnectionPool(host='datatalks.club', port=443): Max retries exceeded with url: /faq/json/courses.json (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1081)')))

In [ ]:
# Defining the tool

# First we define a top-level search function that queries the index directly. The model will reference it by this name. We keep the Python function and the tool name aligned so the dispatch is easier later.

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:


# Next we tell the model about this function. The model doesn't see our Python code, only a schema describing what the function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.

search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
        }
    }
    
}

In [11]:
# Sending the question with the tool

# Now we send the same question as before, but this time we include the tool in the request:

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=message1,
    tools=[search_tool],
)

response.choices[0].message.content

In [ ]:
# Executing the function and sending the result back

# The function call contains JSON arguments. We parse them, call our search function, and serialize the result.

import json

tool_calls =response.choices[0].message.tool_calls

call = tool_calls[0].function

args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)


In [ ]:
# Now we send this result back to the model. First, we add the model's output to the conversation history - the model needs to see its own function call. Then we add the tool result.

message1.extend(response.output)

message1.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [ ]:
# Asking the model again

# We call the API a second time with the expanded history:

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=message1,
    tools=[search_tool],
)

response.choices[0].message.content

In [ ]:
# Token usage and cost

# We just made two API calls instead of one. Each call we send to the model costs money, so it's worth checking how much one tool-using turn actually costs.

# The response has a usage field with the token counts:

usage = response.usage
usage.input_tokens, usage.output_tokens

In [ ]:
# For each model the provider publishes a price per million input tokens and per million output tokens. Plug those numbers in to convert tokens to dollars.


def calculate_model_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_model_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))